In [1]:
from dotenv import load_dotenv
import os
import boto3
import io
import csv
import re
import pandas as pd
from openpyxl import load_workbook
import pyarrow as pa
import pyarrow.parquet as pq

# Load credentials from .env file
load_dotenv()

os.environ['AWS_ACCESS_KEY_ID'] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = os.getenv('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = os.getenv('AWS_DEFAULT_REGION')

In [2]:
s3 = boto3.client("s3")

# COMMON FUNCTION: EXCEL → PARQUET
def excel_to_parquet(input_bucket, input_key, output_bucket, output_key):
    obj = s3.get_object(Bucket=input_bucket, Key=input_key)
    excel_stream = io.BytesIO(obj["Body"].read())

    wb = load_workbook(excel_stream, read_only=True, data_only=True)
    ws = wb.active

    rows = ws.iter_rows(values_only=True)

    # CLEAN + DEDUPLICATE HEADERS
    raw_header = list(next(rows))
    raw_header[0] = "MonthYear"  # safe rename (even if not used later)

    clean_cols = []
    seen = {}

    for col in raw_header:
        if col is None or (isinstance(col, float) and pd.isna(col)):
            col = "unnamed"

        col = (
            str(col)
            .strip()
            .lower()
            .replace(" ", "_")
            .replace("%", "pct")
        )
        col = re.sub(r"[^a-z0-9_]", "", col)

        if col in seen:
            seen[col] += 1
            col = f"{col}_{seen[col]}"
        else:
            seen[col] = 0

        clean_cols.append(col)

    # CREATE DATAFRAME
    df = pd.DataFrame(rows, columns=clean_cols)
    wb.close()

    # FORCE ALL COLUMNS TO STRING
    df = df.astype(str)

    # WRITE PARQUET
    table = pa.Table.from_pandas(df, preserve_index=False)

    parquet_buffer = io.BytesIO()
    pq.write_table(table, parquet_buffer)

    s3.put_object(
        Bucket=output_bucket,
        Key=output_key,
        Body=parquet_buffer.getvalue()
    )

    print(f"✅ Parquet written: s3://{output_bucket}/{output_key}")


In [ ]:
# CAMPAIGN DATA
excel_to_parquet(
    input_bucket="titan-glue-test-data",
    input_key="client_data_excel/1_12082025/campaign_data/Campaign_Tracker_Updated.xlsx",
    output_bucket="titan-glue-test-data",
    output_key="initial_client_data_parquet/campaign_data/campaign_data.parquet"
)

# MEMBER DATA
excel_to_parquet(
    input_bucket="titan-glue-test-data",
    input_key="client_data_excel/1_12082025/member_data/TitanUAE_EnrolmentReport_08122025.xlsx",
    output_bucket="titan-glue-test-data",
    output_key="initial_client_data_parquet/member_data/member_data.parquet"
)

# TRANSACTION DATA
excel_to_parquet(
    input_bucket="titan-glue-test-data",
    input_key="client_data_excel/1_12082025/transaction_data/UAETransactionReportfrominceptiontill_08122025.xlsx",
    output_bucket="titan-glue-test-data",
    output_key="initial_client_data_parquet/transaction_data/transaction_data.parquet"
)


✅ Parquet written: s3://titan-glue-test-data/initial_client_data_parquet/member_data/member_data.parquet
✅ Parquet written: s3://titan-glue-test-data/initial_client_data_parquet/transaction_data/transaction_data.parquet
